<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>LaSDI NN-AE - DIMENSIONALITY REDUCTION + STATIC MAPPING</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import math
import time
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt

from itertools import combinations_with_replacement

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor)

In [ ]:
# Set random seed for reproducibility
seed = 10
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
inp_neuron   = x_dim*y_dim  #Nfile - 1
epochs       = 2000
batchsize    = 256 
latent_dim   = 4 

learn_rate   = 1e-2
step_epoch   = 200
decay_rate   = 0.50

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_data_256_Re_100_0.vtk"
data_file    = "Grid_data_256_Re_"
Nfile        = 1000                                      # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../../")
directory    = os.getcwd()
path         = directory + "/Data_generation/Re_100/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xmin, xmax =  1.0, 20
ymin, ymax = -8.0, 8.0

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))

x = (x - xmin) / (xmax - xmin)                 # [0  to 1 ]
y = (2.0*y - ymin - ymax) / (ymax - ymin)      # [-1 to 1 ]


print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
directory  = os.getcwd()
fieldname  = 'norm_vort'

vort            = np.zeros((Nfile, x_dim, y_dim), dtype=np.float32)

path = os.path.join(directory, "Data_generation", f"Re_100")
for i in range(Nfile):
    file_name = os.path.join(path, f"{data_file}{int(100)}_{i}.vtk")
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader() 
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput()   
    pointData = data.GetPointData().GetArray(fieldname)    
    vort    [i, :, :]  = np.reshape(vtk_to_numpy(pointData), (x_dim, y_dim))
    print("*" * 85)


<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS - AE
</div>


In [ ]:
class ENCODER(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inp_neuron, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, latent_dim)   # final bottleneck = 256
        )
    def forward(self, x):
        return self.encoder(x)

class DECODER(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, inp_neuron),
            # optionally add a final activation depending on input scale:
            # nn.Sigmoid()  # if inputs scaled to [0,1]
        )
    def forward(self, x):
        return self.decoder(x)

def init_weights(m):
    if isinstance(m, nn.Linear):
        # Xavier for symmetric activations / MSE
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
        
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort).to(processor).reshape(vort.shape[0], -1)
vort_output  = torch.Tensor(vort).to(processor).reshape(vort.shape[0], -1)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

dataset               = TensorDataset(vort_input, vort_output)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10,        shuffle=False)
plot_loader    = DataLoader(dataset,    batch_size=batchsize, shuffle=False)
    
encoder_model  = ENCODER ().to(processor)
decoder_model  = DECODER ().to(processor)

encoder_optim  = optim.Adam(encoder_model.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
decoder_optim  = optim.Adam(decoder_model.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)

encoder_sched  = optim.lr_scheduler.StepLR(encoder_optim, step_size=step_epoch, gamma=decay_rate)
decoder_sched  = optim.lr_scheduler.StepLR(decoder_optim, step_size=step_epoch, gamma=decay_rate)

encoder_params = count_params(encoder_model)
decoder_params = count_params(decoder_model)

print(f"Encoder Parameters: {encoder_params}")
print(f"Decoder Parameters: {decoder_params}")


<div style="text-align: center; font-weight: bold; font-size: 1em;">
    AE - TRAINING DATA
</div>


In [ ]:
Loss_Data   = torch.empty(size=(epochs, 1))
loss_func   = nn.MSELoss()

for epoch in range (epochs):
    total_loss = 0.0
    for batch_idx, (w_in, w_out) in enumerate(train_loader):
        encoder_output = encoder_model (w_in)
        decoder_output = decoder_model (encoder_output)
        batch_loss     = loss_func   (decoder_output, w_out)

        encoder_optim.zero_grad()
        decoder_optim.zero_grad()
        batch_loss.backward()

        encoder_optim.step()
        decoder_optim.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', encoder_optim.param_groups[0]['lr'])
    print ("*"*85)
    
    encoder_sched.step()
    decoder_sched.step()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(encoder_model.state_dict(), "AE_ENCODER_MODEL_4.pt" )
torch.save(decoder_model.state_dict(), "AE_DECODER_MODEL_4.pt" )
torch.save(Loss_Data  [0:epoch],  "AE_Loss_4.pt"  )

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        FNO - TESTING DATA
</div>


In [ ]:
# TEST ERROR OF FNO
encoder_model.eval()
decoder_model.eval()

total_loss = 0.0
for batch_idx, (input_data, output_data) in enumerate(test_loader):
    encoder_output_data = encoder_model (input_data)
    decoder_output_data = decoder_model (encoder_output_data)
    batch_loss          = loss_func (decoder_output_data, output_data)

    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())

In [ ]:
# ── 2a. SINDy candidate library ───────────────────────────────
def sindy_library(z, poly_order=2, include_sine=False):
    """
    z           : (N, latent_dim)  — rows are individual latent states
    poly_order  : highest polynomial degree to include
    returns
      Theta : (N, n_terms)   candidate feature matrix
      labels: list[str]      human-readable term names (for inspecting Xi)
    """
    N, n = z.shape
    device = z.device

    cols, labels = [], []

    # constant
    cols.append(torch.ones(N, 1, device=device))
    labels.append('1')

    # degree 1
    for i in range(n):
        cols.append(z[:, i:i+1])
        labels.append(f'z{i}')

    # degree 2 … poly_order
    for order in range(2, poly_order + 1):
        for idx in combinations_with_replacement(range(n), order):
            term = torch.ones(N, device=device)
            name = ''
            for i in idx:
                term = term * z[:, i]
                name += f'z{i}'
            cols.append(term.unsqueeze(1))
            labels.append(name)

    # optional trig
    if include_sine:
        for i in range(n):
            cols.append(torch.sin(z[:, i:i+1]))
            cols.append(torch.cos(z[:, i:i+1]))
            labels += [f'sin(z{i})', f'cos(z{i})']

    return torch.cat(cols, dim=1), labels   # (N, n_terms)


def sindy_n_terms(latent_dim, poly_order=2, include_sine=False):
    from math import comb
    n = latent_dim
    size = 1 + n
    for o in range(2, poly_order + 1):
        size += comb(n + o - 1, o)
    if include_sine:
        size += 2 * n
    return size


# ── 2b. Finite differences on a latent trajectory ─────────────
def finite_difference_trajectory(z_traj, dt):
    """
    z_traj : (T, latent_dim)  — ordered time snapshots for ONE trajectory
    dt     : scalar time step (assumed uniform)
    returns dz_dt : (T, latent_dim)
    """
    dz = torch.zeros_like(z_traj)
    dz[1:-1] = (z_traj[2:] - z_traj[:-2]) / (2.0 * dt)  # central
    dz[0]    = (z_traj[1]  - z_traj[0])   / dt           # forward
    dz[-1]   = (z_traj[-1] - z_traj[-2])  / dt           # backward
    return dz


# ── 2c. Sequential-threshold least-squares (STRidge) ──────────
def stridge(Theta, dz_dt, threshold=1e-3, max_iter=10, ridge_alpha=1e-4):
    """
    Solves  dz_dt ≈ Theta @ Xi  via iterative hard thresholding.

    Theta   : (N, n_terms)
    dz_dt   : (N, latent_dim)
    returns
      Xi    : (n_terms, latent_dim)  sparse coefficient matrix
    """
    n_terms   = Theta.shape[1]
    latent_dim = dz_dt.shape[1]
    device    = Theta.device

    # initialise with ridge regression  Xi = (TᵀT + αI)⁻¹ Tᵀ dz
    A   = Theta.T @ Theta + ridge_alpha * torch.eye(n_terms, device=device)
    rhs = Theta.T @ dz_dt
    Xi  = torch.linalg.solve(A, rhs)          # (n_terms, latent_dim)

    # sequential thresholding
    mask = torch.ones_like(Xi, dtype=torch.bool)
    for _ in range(max_iter):
        mask = Xi.abs() >= threshold
        # refit only active terms per latent dimension
        for k in range(latent_dim):
            active = mask[:, k]
            if active.sum() == 0:
                Xi[:, k] = 0.0
                continue
            Th_k = Theta[:, active]            # (N, n_active)
            A_k  = Th_k.T @ Th_k + ridge_alpha * torch.eye(active.sum(), device=device)
            rhs_k = Th_k.T @ dz_dt[:, k]
            Xi[active, k]  = torch.linalg.solve(A_k, rhs_k)
            Xi[~active, k] = 0.0

    return Xi                                   # (n_terms, latent_dim)


# ── 2d. LASDi main class ───────────────────────────────────────
class LASDi:
    """
    Wraps your pre-trained ENCODER / DECODER and adds:
      - latent trajectory extraction
      - SINDy identification (STRidge)
      - RK4 latent-space integration
      - optional decoder fine-tuning on SINDy-generated trajectories
    """

    def __init__(self, encoder_model, decoder_model,
                 latent_dim, poly_order=2, include_sine=False,
                 device='cpu'):
        self.encoder    = encoder_model.to(device)
        self.decoder    = decoder_model.to(device)
        self.latent_dim = latent_dim
        self.poly_order = poly_order
        self.inc_sine   = include_sine
        self.device     = device

        self.Xi     = None   # (n_terms, latent_dim) — set after identify()
        self.labels = None   # term names

        # freeze encoder weights — stage 2 does NOT update them
        for p in self.encoder.parameters():
            p.requires_grad_(False)
        self.encoder.eval()

    # ── encode a batch of snapshots ──────────────────────────
    @torch.no_grad()
    def encode_trajectory(self, x_traj):
        """
        x_traj : (T, inp_neuron)  or  (T, B, inp_neuron)
        returns z_traj of same leading shape with last dim = latent_dim
        """
        shape = x_traj.shape
        flat  = x_traj.reshape(-1, shape[-1]).to(self.device)
        z_flat = self.encoder(flat)
        return z_flat.reshape(*shape[:-1], self.latent_dim)

    # ── stage 2 fit ───────────────────────────────────────────
    def identify(self, trajectories, dt,
                 threshold=1e-3, max_iter=10, ridge_alpha=1e-4):
        """
        trajectories : list of tensors, each (T_i, inp_neuron)
                       OR a single tensor (T, inp_neuron)
        dt           : time step (float)

        Encodes every trajectory → computes FD derivatives →
        stacks everything → runs STRidge → stores self.Xi
        """
        if isinstance(trajectories, torch.Tensor):
            trajectories = [trajectories]

        all_z, all_dz = [], []
        for traj in trajectories:
            z  = self.encode_trajectory(traj)          # (T, latent_dim)
            dz = finite_difference_trajectory(z, dt)   # (T, latent_dim)
            all_z.append(z)
            all_dz.append(dz)

        Z  = torch.cat(all_z,  dim=0).to(self.device)  # (N_total, latent_dim)
        DZ = torch.cat(all_dz, dim=0).to(self.device)

        Theta, self.labels = sindy_library(Z, self.poly_order, self.inc_sine)
        self.Xi = stridge(Theta, DZ, threshold, max_iter, ridge_alpha)

        n_active = (self.Xi.abs() > 0).sum().item()
        print(f'[LASDi] SINDy identified.  Active terms: {n_active} / {self.Xi.numel()}')
        return self.Xi.cpu()

    # ── predict dz/dt for given z ─────────────────────────────
    @torch.no_grad()
    def sindy_rhs(self, z):
        """z : (B, latent_dim)  →  dz/dt : (B, latent_dim)"""
        Theta, _ = sindy_library(z, self.poly_order, self.inc_sine)
        return Theta @ self.Xi

    # ── RK4 integration in latent space ──────────────────────
    @torch.no_grad()
    def integrate(self, z0, n_steps, dt):
        """
        z0      : (1, latent_dim) or (latent_dim,)  initial latent state
        n_steps : int
        dt      : float
        returns
          z_traj : (n_steps+1, latent_dim)
          x_traj : (n_steps+1, inp_neuron)
        """
        assert self.Xi is not None, "Call .identify() before .integrate()"
        self.decoder.eval()

        z = z0.reshape(1, self.latent_dim).to(self.device)
        z_traj = [z.cpu()]

        for _ in range(n_steps):
            k1 = self.sindy_rhs(z)
            k2 = self.sindy_rhs(z + 0.5 * dt * k1)
            k3 = self.sindy_rhs(z + 0.5 * dt * k2)
            k4 = self.sindy_rhs(z +       dt * k3)
            z  = z + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
            z_traj.append(z.cpu())

        z_traj = torch.cat(z_traj, dim=0)                    # (n_steps+1, latent_dim)
        x_traj = self.decoder(z_traj.to(self.device)).cpu()  # (n_steps+1, inp_neuron)
        return z_traj, x_traj

    # ── optional: fine-tune decoder on SINDy trajectories ────
    def finetune_decoder(self, trajectories, dt,
                         n_steps_per_ic=None, epochs=50, lr=1e-4):
        """
        After SINDy is fitted, generate predicted trajectories and
        fine-tune only the DECODER so decoded predictions match data.

        trajectories : list of (T, inp_neuron) tensors (same as identify)
        n_steps_per_ic : if None uses output trajectory length
        """
        assert self.Xi is not None, "Call .identify() first"

        # decoder unfrozen; encoder stays frozen
        optim = torch.optim.Adam(self.decoder.parameters(), lr=lr)
        mse   = nn.MSELoss()

        for epoch in range(epochs):
            total_loss = 0.0
            for traj in trajectories:
                T   = traj.shape[0]
                n_s = (T - 1) if n_steps_per_ic is None else n_steps_per_ic

                # get initial latent state from real data
                z0 = self.encode_trajectory(traj[0:1])          # (1, latent_dim)

                # integrate forward with SINDy
                _, x_pred = self.integrate(z0, n_s, dt)         # (n_s+1, inp_neuron)

                x_true = traj[:n_s+1].to(self.device)
                x_pred = x_pred.to(self.device)

                loss = mse(x_pred, x_true)
                optim.zero_grad()
                loss.backward()
                optim.step()
                total_loss += loss.item()

            if (epoch + 1) % 10 == 0:
                print(f'  [finetune] epoch {epoch+1:4d}  loss {total_loss/len(trajectories):.6f}')

    # ── print discovered equations ────────────────────────────
    def print_equations(self, threshold=1e-4):
        assert self.Xi is not None and self.labels is not None
        print('\n[LASDi] Discovered latent ODEs:')
        for k in range(self.latent_dim):
            terms = []
            for j, label in enumerate(self.labels):
                val = self.Xi[j, k].item()
                if abs(val) > threshold:
                    terms.append(f'{val:+.4f}·{label}')
            rhs = '  '.join(terms) if terms else '0'
            print(f'  dz{k}/dt = {rhs}')


In [ ]:
configs = [0.5, 1.5, 2.0, 2.5, 3.0, 5.0]
dt = 0.01
x_true  = vort_output.detach().cpu().numpy()
N = vort_output.shape[0]
t = np.arange(N) * dt

N = vort_output.shape[0]
print(f"N={N}, vort_output.shape={vort_output.shape}")
x_true = vort_output.detach().cpu().numpy()
print(f"x_true.shape={x_true.shape}")
results = {}

# ══════════════════════════════════════════════════════════════
# ROLLOUT LOOP
# ══════════════════════════════════════════════════════════════
for thresh in configs:
    lasdi_test = LASDi(encoder_model, decoder_model,
                       latent_dim=latent_dim,
                       poly_order=2,
                       include_sine=False,
                       device=processor)
    lasdi_test.identify([vort_output], dt=dt,
                        threshold=thresh,
                        max_iter=10,
                        ridge_alpha=1e-4)
    # ── print equations for this config ──────────────────────
    print(f'\n{"="*60}')
    print(f'  thresh={thresh}')
    print(f'{"="*60}')
    lasdi_test.print_equations()

    x_rollout   = torch.zeros(N, vort_output.shape[1])
    z_rollout   = torch.zeros(N, latent_dim)
    blowup_step = N

    start_time = time.time()
    with torch.no_grad():
        z_current    = lasdi_test.encode_trajectory(vort_output[0:1])
        z_rollout[0] = z_current.cpu()
        x_rollout[0] = decoder_model(z_current).cpu()

        for n in range(N - 1):
            k1 = lasdi_test.sindy_rhs(z_current)
            k2 = lasdi_test.sindy_rhs(z_current + 0.5 * dt * k1)
            k3 = lasdi_test.sindy_rhs(z_current + 0.5 * dt * k2)
            k4 = lasdi_test.sindy_rhs(z_current +       dt * k3)
            z_current = z_current + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

            if torch.isnan(z_current).any() or z_current.abs().max() > 1e4:
                if blowup_step == N:
                    blowup_step = n + 1
                z_current = torch.nan_to_num(z_current, nan=0.0)
                z_current = torch.clamp(z_current, -1e4, 1e4)

            z_rollout[n + 1] = z_current.cpu()
            x_rollout[n + 1] = decoder_model(z_current).cpu()

    x_pred   = np.nan_to_num(x_rollout.numpy(), nan=0.0)
    err_l2   = (np.linalg.norm(x_true - x_pred, axis=1) /
               (np.linalg.norm(x_true,           axis=1) + 1e-10))
    err_mean = np.abs(x_true - x_pred).mean(axis=1)
    err_mse  = ((x_true - x_pred) ** 2).mean(axis=1)
    active   = (lasdi_test.Xi.abs() > 0).sum().item()

    elapsed_hours = (time.time() - start_time) / 3600
    print(elapsed_hours)

    label = f'thresh={thresh}  active={active}'
    results[label] = {
        'err_l2':       err_l2,
        'err_mean':     err_mean,
        'err_mse':      err_mse,
        'z':            z_rollout.numpy(),
        'blowup_step':  blowup_step,
        'thresh':       thresh,
        'active':       active,
    }
    blowup_str = f'blowup@step={blowup_step}' if blowup_step < N else 'stable'
    print(f'thresh={thresh}  active={active:3d}  {blowup_str}  '
          f'mean_L2={err_l2.mean():.4f}  '
          f'final_L2={err_l2[-1]:.4f}  '
          f'mean_MSE={err_mse.mean():.4f}')

# ══════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════
cmap = plt.cm.get_cmap('tab10', len(results))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('LASDi — threshold comparison (clipped rollout)', fontsize=14, fontweight='bold')

for i, (label, res) in enumerate(results.items()):
    color = cmap(i)
    bs = res['blowup_step']

    axes[0].semilogy(t[:bs],   res['err_mean'][:bs],  color=color, linewidth=1.5, label=label)
    axes[0].semilogy(t[bs-1:], res['err_mean'][bs-1:], color=color, linewidth=1.0, linestyle='--')

    axes[1].semilogy(t[:bs],   res['err_mse'][:bs],    color=color, linewidth=1.5, label=label)
    axes[1].semilogy(t[bs-1:], res['err_mse'][bs-1:],   color=color, linewidth=1.0, linestyle='--')

    axes[2].semilogy(t[:bs],   res['err_l2'][:bs],     color=color, linewidth=1.5, label=label)
    axes[2].semilogy(t[bs-1:], res['err_l2'][bs-1:],    color=color, linewidth=1.0, linestyle='--')

axes[0].set_ylabel('Mean abs error')
axes[0].set_title('Mean absolute error  (solid=stable  dashed=after blowup)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('MSE')
axes[1].set_title('Mean squared error')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel('Relative L2')
axes[2].set_xlabel('Time')
axes[2].set_title('Relative L2  ||true - pred|| / ||true||')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# TECPLOT .DAT EXPORT
# ══════════════════════════════════════════════════════════════
def write_tecplot_dat(filename, t, data_dict, zone_title='LASDi'):
    var_names  = list(data_dict.keys())
    var_values = list(data_dict.values())
    N          = len(t)
    with open(filename, 'w') as f:
        varlist = ' '.join([f'"{v}"' for v in ['Time'] + var_names])
        f.write(f'TITLE = "{zone_title}"\n')
        f.write(f'VARIABLES = {varlist}\n')
        f.write(f'ZONE T="{zone_title}", I={N}, F=POINT\n')
        for i in range(N):
            row = [t[i]] + [v[i] for v in var_values]
            f.write('  '.join(f'{val:.8e}' for val in row) + '\n')
    print(f'Written: {filename}  ({N} rows)')

os.makedirs('lasdi_results', exist_ok=True)

for (label, res) in results.items():
    thresh = res['thresh']
    active = res['active']
    safe   = f'thresh{str(thresh).replace(".","p")}_active{active}'

    # ── error file ────────────────────────────────────────────
    write_tecplot_dat(
        filename   = f'lasdi_results/lasdi_error_{safe}.dat',
        t          = t,
        data_dict  = {
            'err_mean':    res['err_mean'],
            'err_mse':     res['err_mse'],
            'err_l2':      res['err_l2'],
            'blowup_flag': np.array([1.0 if i >= res['blowup_step']
                                     else 0.0 for i in range(N)]),
        },
        zone_title = f'LASDi error {label}'
    )

    # ── latent trajectory file ────────────────────────────────
    z      = res['z']   # (N, latent_dim)
    z_dict = {f'z{i}': z[:, i] for i in range(z.shape[1])}
    write_tecplot_dat(
        filename   = f'lasdi_results/lasdi_latent_{safe}.dat',
        t          = t,
        data_dict  = z_dict,
        zone_title = f'Latent trajectory {label}'
    )

print('\nAll files written to lasdi_results/')

In [ ]:
output_path = 'lasdi_rollout_error.dat'

with open(output_path, 'w') as f:
    # Tecplot header
    f.write('TITLE = "LASDi Rollout Error Evolution"\n')
    f.write('VARIABLES = "Time" "MSE" "Mean_Abs_Error" "Max_Abs_Error" "Relative_L2"\n')
    f.write(f'ZONE T="Rollout", I={N}, F=POINT\n')
    
    for i in range(N):
        f.write(f'{t[i]:.6e}  '
                f'{error_mean[i]:.6e}  '
                f'{error_max[i]:.6e}  '
                f'{error_l2[i]:.6e}\n')

print(f'Written: {output_path}  ({N} rows)')